In [15]:
import numpy as np
import random
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [1]:
def generar_red(n_nodos: int, max_hijos: int = 5):
    """
    Genera una red ramificada tipo árbol:
    - Un nodo raíz: 'Matriz_Principal'
    - Cada nodo puede tener entre 1 y max_hijos hijos (mientras queden nodos)
    Retorna:
        red_estructura: dict {hijo: padre}
        nodos_finales: lista de nodos hoja (sin hijos)
        lista_nodos: lista ordenada de nodos (para indexar matrices)
    """
    if n_nodos < 1:
        raise ValueError("n_nodos debe ser al menos 1")

    red_estructura = {'Matriz_Principal': None}
    nodos_creados = ['Matriz_Principal']
    frontera = ['Matriz_Principal']
    siguiente_id = 1

    # Vamos creando hijos mientras falten nodos
    while len(nodos_creados) < n_nodos and frontera:
        padre = frontera.pop(0)

        restantes = n_nodos - len(nodos_creados)
        if restantes <= 0:
            break

        max_hijos_posibles = min(max_hijos, restantes)
        n_hijos = random.randint(1, max_hijos_posibles)

        for _ in range(n_hijos):
            nombre_hijo = f"Nodo_{siguiente_id}"
            siguiente_id += 1

            red_estructura[nombre_hijo] = padre
            nodos_creados.append(nombre_hijo)
            frontera.append(nombre_hijo)

            if len(nodos_creados) >= n_nodos:
                break

    # Nodos finales (hojas)
    padres = set(red_estructura.values())
    nodos_finales = [n for n in red_estructura.keys() if n not in padres]

    lista_nodos = list(red_estructura.keys())

    return red_estructura, nodos_finales, lista_nodos


In [2]:
def generar_adjacencia(red_estructura: dict, lista_nodos: list):
    """
    Genera la matriz de adyacencia a partir de la estructura de la red.
    Retorna:
        matriz_adyacencia: np.ndarray
    """
    n = len(lista_nodos)
    matriz_adyacencia = np.zeros((n, n), dtype=int)
    indice_nodo = {nodo: i for i, nodo in enumerate(lista_nodos)}

    for hijo, padre in red_estructura.items():
        if padre is not None:
            i = indice_nodo[padre]
            j = indice_nodo[hijo]
            matriz_adyacencia[i, j] = 1  # Conexión del padre al hijo

    return matriz_adyacencia

In [5]:
estructura, nodos_finales, lista_nodos = generar_red(100)

In [20]:
matriz_adyacencia = generar_adjacencia(estructura, lista_nodos)

In [21]:
matriz_adyacencia

array([[0, 1, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(100, 100))

In [7]:
nodo_fuga = random.choice(lista_nodos)
while nodo_fuga in nodos_finales:
    nodo_fuga = random.choice(lista_nodos)

print(f"Nodo de fuga seleccionado: {nodo_fuga}")

Nodo de fuga seleccionado: Nodo_26


In [8]:
# --- 1. FUNCIÓN AUXILIAR: Obtener Orden Topológico Inverso (Post-Orden) ---
def orden_desde_hojas(nodo, estructura):
    orden = []

    por_visitar = []

    for hijo, padre in estructura.items():
        if nodo == padre:
            por_visitar.append(hijo)


    for hijo in por_visitar:
        orden.extend(orden_desde_hojas(hijo, estructura))

    return orden + [nodo]

In [9]:
caudales = np.zeros(len(lista_nodos))
orden_revisar = orden_desde_hojas("Matriz_Principal", estructura)


# Simular caudales nodos finales
for nodo in nodos_finales:
    indice = lista_nodos.index(nodo)
    caudales[indice] = random.uniform(1.0, 10.0)  # Caudal aleatorio entre 1 y 10


for nodo in orden_revisar:
    padre = estructura.get(nodo)
    if padre is None:
        continue

    if nodo == nodo_fuga:
        caudal_fuga = random.uniform(20.0, 50.0)  # Fuga entre 20 y 50
        caudales[lista_nodos.index(nodo)] += caudal_fuga
        print(f"Nodo {nodo} tiene una fuga de {caudal_fuga}")

    indice_padre = lista_nodos.index(padre)
    caudales[indice_padre] += caudales[lista_nodos.index(nodo)]

Nodo Nodo_26 tiene una fuga de 40.15354069219491


In [10]:
ingreso_total = caudales[lista_nodos.index('Matriz_Principal')]
ingreso_total

np.float64(388.6875138276896)

In [11]:
salida_total = np.sum(caudales[[lista_nodos.index(n) for n in nodos_finales]])
salida_total

np.float64(348.53397313549465)

In [12]:
ingreso_total - salida_total

np.float64(40.15354069219495)

In [13]:
caudales

array([388.68751383, 255.23353293,  87.2694361 ,  33.07804304,
        13.10650176,  41.30037057,  52.87042305, 139.70150049,
        21.36123881,  26.29027236,  39.7602991 ,  17.54055069,
         3.67831395,   5.44635468,  14.48631231,  13.14537605,
        13.10650176,  11.20257739,  30.09779318,  13.5761837 ,
        17.96155439,  21.33268496,  24.57254084,  18.36968877,
        15.7954656 ,  19.88989235,  61.07391292,   6.47775147,
        10.73510054,   4.1483868 ,   7.41989582,   7.4118022 ,
        11.45857433,  29.50157866,   1.79333778,   8.46538266,
         4.77237742,   7.2416072 ,   3.86340887,   1.6631572 ,
         3.67831395,   4.00853607,   1.43781861,   7.38842345,
         7.09788886,   9.61088687,   3.53448919,   1.29446258,
         1.04331824,   1.61872235,   9.1499986 ,   5.64114293,
         3.92156799,   1.63986647,   8.47718586,   6.02340917,
         6.09431851,   9.50287964,   5.25027309,   8.32591061,
         2.90076443,   2.77441769,   6.26641431,   6.01

In [14]:
# Revisar red desde padre a hijo para detectar inconsistencias
nodos_visitar = ['Matriz_Principal']

while nodos_visitar:
    nodo = nodos_visitar.pop(0)

    caudal_nodo = caudales[lista_nodos.index(nodo)]
    hijos = [hijo for hijo, padre in estructura.items() if padre == nodo]
    caudal_hijos = sum(caudales[lista_nodos.index(h)] for h in hijos)

    if abs(caudal_nodo - caudal_hijos) > 1e-6:
        print(f"Inconsistencia en {nodo}: caudal nodo = {caudal_nodo}, suma hijos = {caudal_hijos}")
        print(f"Posible fuga entre {nodo} y {hijos} de {caudal_nodo - caudal_hijos}")
        break

    nodos_visitar.extend(hijos)

Inconsistencia en Nodo_26: caudal nodo = 61.073912918163586, suma hijos = 20.920372225968677
Posible fuga entre Nodo_26 y ['Nodo_80', 'Nodo_81', 'Nodo_82', 'Nodo_83'] de 40.15354069219491


In [ ]:
# Graficar la red con layout jerárquico sin solapamiento
def graficar_matriz(matriz_adyacencia: np.ndarray, lista_nodos: list = None, title: str = "Gráfico de Red General"):
    """
    Convierte una matriz de adyacencia general en un grafo dirigido y lo visualiza
    usando un layout de niveles (Y) y un índice de posición (X) calculado para 
    garantizar que no haya solapamiento horizontal.

    Args:
        adj_matrix (np.ndarray): Matriz de adyacencia (N x N).
        node_names (list, optional): Lista de nombres para las etiquetas. Si es None, usa índices (0, 1, ...).
        title (str): Título del gráfico.
    """
    G = nx.DiGraph(matriz_adyacencia) 
    N = matriz_adyacencia.shape[0]

    # 1. Identificar la Raíz (Nodo con in-degree = 0)
    in_degree = G.in_degree()
    root_candidates = [node for node, degree in in_degree if degree == 0]
    root_node = root_candidates[0] if root_candidates else 0

    # 2. Calcular las distancias y agrupar nodos por nivel (distancia)
    try:
        distancia_desde_raiz = nx.shortest_path_length(G, source=root_node)
    except nx.NetworkXNoPath:
        distancia_desde_raiz = {node: 0 for node in G.nodes()}

    # Agrupar nodos por nivel: {distancia: [lista de nodos]}
    nodos_por_nivel = {}
    for node, dist in distancia_desde_raiz.items():
        if dist not in nodos_por_nivel:
            nodos_por_nivel[dist] = []
        nodos_por_nivel[dist].append(node)

    # 3. Generar las posiciones (X y Y) sin solapamiento
    pos = {}
    
    # Factor de separación vertical (Y) y horizontal (X)
    Y_SEP = 3
    X_SEP = 1.5

    for dist, nodes in nodos_por_nivel.items():
        num_nodes_in_level = len(nodes)
        
        # Calcular el desplazamiento inicial para centrar el nivel
        start_x = - (num_nodes_in_level - 1) * X_SEP / 2
        
        for i, node in enumerate(nodes):
            # Posición Y: Inverso de la distancia (Top-Down) * factor de separación
            y = -dist * Y_SEP
            
            # Posición X: Índice del nodo en su nivel * factor de separación, ajustado para centrar
            x = start_x + i * X_SEP
            
            pos[node] = (x, y)

    # 4. Asignar etiquetas
    labels = lista_nodos if lista_nodos else {i: str(i) for i in range(N)}
    
    # --- 2. ASIGNACIÓN DE COLORES DINÁMICA ---
    # Si la fuga es > 0, el color es rojo. Si no hay fuga o es la raíz, es azul (teal).
    colores_nodos = []
    for i in range(N):
        if caudal_fuga[i] > 0:
            colores_nodos.append('red')
        else:
            colores_nodos.append('teal')

    # 3. Dibujar el grafo
    labels = lista_nodos if lista_nodos else {i: str(i) for i in range(N)}
    
    plt.figure(figsize=(12, 8))
    
    # Usar la lista de colores generada
    nx.draw_networkx_nodes(G, pos, node_color=colores_nodos, node_size=1500, alpha=0.9)
    
    nx.draw_networkx_edges(G, pos, edgelist=G.edges(), arrowstyle='->', 
                           arrowsize=20, edge_color='gray', width=1.5)
    
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=12, font_color='white', font_weight='bold')
    
    # 5. Dibujar el grafo
    plt.figure(figsize=(12, 8))
    
    nx.draw_networkx_nodes(G, pos, node_color='teal', node_size=1500, alpha=0.9)
    
    nx.draw_networkx_edges(G, pos, edgelist=G.edges(), arrowstyle='->', 
                           arrowsize=20, edge_color='gray', width=1.5)
    
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=12, font_color='white', font_weight='bold')
    
    plt.title(title, fontsize=16)
    plt.axis('off') 
    plt.tight_layout()
    plt.savefig("grafo_general_jerarquico_sin_solapamiento.png")
    plt.close()

In [19]:
graficar_matriz(matriz_adyacencia, node_names=lista_nodos, title="Red de Distribución con Fuga")

TypeError: 'float' object is not subscriptable

## Solución 1: Balance de masas en la red (Detectar si lo que entra sea igual o diferente a lo que sale de la red).

## Solución 2: Localizar la fuga con mayor probabilidad.

## Solución 3: Monitoreo de redes de agua y simulaciones de arreglos y fugas.